In [3]:
import ast
import os
import optuna
import ipynbname
import pandas as pd
import numpy as np
from optuna.exceptions import TrialPruned
from Functions.AutoCloud_V4_5 import *
from Functions.TedaOptimize_V4_5 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT_R2 import *
import copy

FileName = ipynbname.name()

# Configuração do experimento
FEEDBACK = 'symmetric'    # 'symmetric' (gradiente no passo atual) ou 'random' (ablação: RFLO + DFA)
WTAG, WTAP = False, False # configuração principal: soma ponderada dos grânulos (eq. 36)
EOL = 0.3                 # limiar de fim de vida (EoL) do HI
W = 1                       # janela do DSI em cada X_k (nº de amostras do RS; PrepareData/WindowDSI)

# rho dinâmico (AutoCloud_V4_2): começa em 0 e é ajustado a cada iteração maximizando V = média(cov)*média(sp)
RHO_TARGET = 'RUL'           # 'HI' (causal, HI um passo à frente) ou 'RUL' (oráculo: usa a RUL real; n_rho previsões por passo)
RHO_MIN, RHO_MAX, N_RHO = 1e-3, 1, 4   # grade log-espaçada de candidatos
LAM = 1.0                   # fator de esquecimento (1.0 = todos os passos com o mesmo peso)

out_path = os.path.join('Optimization', RunTag(FileName,FEEDBACK,WTAG,WTAP), 'multi', 'Optimization.csv')
dfs = [pd.read_csv(out_path)] if os.path.exists(out_path) else []
RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_excel(r'Dataset\HI.xlsx')

sig = HI['PC1'].values

In [4]:
params = [3.5,28,[70],8,0,1,2,5,19,0]
params = [2.5,33,[32, 49],1,2,8,9,5,2,1,1]
m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX)   # avisa se o fluxo não alcança o EoL
if isinstance(nR, str): nR = ast.literal_eval(nR)

RHO_DYN, RHO_ETA, RHO_T, RHO_BETA = True, 0.1, 0.5, 0.75              # False: rho estático (= RHO), passo, temperatura da cobertura suavizada (escore |e|/e_est), momento
RHO, RHO_MIN, RHO_MAX = 0.5, 1e-2, 10
INTR_STATE = 'reset'       

teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],act=act,
               tau=τ,eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
               rho=RHO,lam=LAM,intr_state=INTR_STATE,
               rho_dyn=RHO_DYN,rho_eta=RHO_ETA,rho_T=RHO_T,rho_beta=RHO_BETA,rho_min=RHO_MIN,rho_max=RHO_MAX,
               rls_mode='pol',nf_rls=3,ff_rls=0.9,dt_rls=1e3,
               )

teda.SimGranularRUL(J,L,X,Y,Z)

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f}")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=3,dsi_mode='hide')


HI (HI_Granularity): cov=0.346 | sp=0.756 | V=média(cov*sp)=0.2780 | Vsum=22.5156


In [32]:
brngs = [f'B1_{i+1}' for i in range(5)]
brngs = ['B1_1','B1_2','B1_3','B1_4','B1_5','B2_1','B2_2','B2_4',]
paths = [f'Dataset\DSIR1n\{brng}.csv' for brng in brngs]
DSIs = [pd.read_csv(path) for path in paths]

teda=AutoCloud(m=1.7,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],act=act,
               tau=τ,eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
               rho=RHO,lam=LAM,intr_state=INTR_STATE,
               rho_dyn=RHO_DYN,rho_eta=RHO_ETA,rho_T=RHO_T,rho_beta=RHO_BETA,rho_min=RHO_MIN,rho_max=RHO_MAX,
               rls_mode='pol',nf_rls=3,ff_rls=0.9,dt_rls=1e3,
               )
w = 2

for DSI in DSIs: 
    teda.SimGranulation(WindowDSI(DSI,w))   

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f}")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide',)

HI (HI_Granularity): cov=0.000 | sp=0.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000
